# Save CSA epoch15 — exact147
Accelerator **None**, Internet ON. Enable Secret KAGGLE_API_TOKEN (or change SECRET_NAME to codex). Downloads only best.bin from exact147, verifies known SHA and creates a new Private Dataset. best/latest15 are identical; one binary is sufficient. Keep original147 output for logs/optimizer. No model execution.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/147'
OWNER='ducanh2403'
SLUG='csa-full-xe15-v147'
TITLE='CSA XE15 v147 preserved weights dataset'
SECRET_NAME='KAGGLE_API_TOKEN' # Set to codex if that is your actual enabled Secret name.
DESTINATION=Path('/kaggle/working/csa_full_xe15_v147_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/147':raise ValueError('This notebook pins147; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/147', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v147', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v147; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json,shutil,tempfile
EXPECTED='e02593ea079bdb563a1afe0c5898abdbf81297aa5e688b3b0df839cb820e5b25'
if DESTINATION.exists():raise RuntimeError('Preserve existing output; fresh destination required.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa147-',dir='/tmp'))
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',r'(^|/)best\.bin$'])
matches=[p for p in DOWNLOAD.rglob('best.bin') if hashlib.file_digest(p.open('rb'),'sha256').hexdigest()==EXPECTED]
if len(matches)!=1:raise RuntimeError('Expected one exact epoch15 best.bin from147; no latest/version fallback.')
DESTINATION.mkdir(parents=True)
shutil.copy2(matches[0],DESTINATION/'best.bin')
if hashlib.file_digest((DESTINATION/'best.bin').open('rb'),'sha256').hexdigest()!=EXPECTED:raise RuntimeError('Copy SHA differs')
(DESTINATION/'checkpoint_receipt.json').write_text(json.dumps(dict(source=KERNEL_VERSION,sha256=EXPECTED,validated_epoch=15,updates=265665,original_bytes=True),indent=2),encoding='utf-8')
print('Exact147 weights copied unchanged; no model loaded.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private byte-preserved CSA epoch15 weights from exact SaveVersion147. Upstream terms retained; no training.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('Creation accepted; Dataset may still be processing. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status unconfirmed; check Dataset page before retrying.',flush=True)


Expected mount: `/kaggle/input/datasets/ducanh2403/csa-full-xe15-v147`. Attach this Dataset to BOTH GPU notebooks; share private Dataset with the second account if required.
